In [105]:
%run ../../langchain_common.py

In [106]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending - echo the body so the model (and final AIMessage) reflects edits
    return f"Email sent with body: {body}"

In [107]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def initialize_new_agent(temperature: float | None = None):
    
    class EmailState(AgentState):
        email: str

    pg_checkpointer = create_pg_checkpointer()

    # Optionally override the model temperature. temperature lives on the model,
    # so bind it here rather than passing it to invoke().
    model = llm_noreason if temperature is None else llm_noreason.bind(temperature=temperature)

    agent = create_agent(
        model=model,
        tools=[read_email, send_email],
        state_schema=EmailState,
        checkpointer=pg_checkpointer,
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "read_email": False,
                    "send_email": True,
                },
                description_prefix="Tool execution requires approval",
            ),
        ],
    )
    return agent

agent = initialize_new_agent()

## Invoking the Agent

The cell below kicks off the agent by calling `agent.invoke()` with:

- A **`HumanMessage`** instructing the agent to read and immediately reply to an email
- The **`email`** field populated in the `EmailState` — this is the email content the `read_email` tool will retrieve from state

The agent will:
1. Call `read_email` (no interrupt configured) → retrieves the email body from state
2. Call `send_email` (interrupt configured) → **pauses here** before sending

Because `HumanInTheLoopMiddleware` is set to interrupt on `send_email`, the agent will **not complete** — it returns a `GraphOutput` containing an `__interrupt__` value that holds the proposed email body for human review before it is sent.

In [108]:
def invoke_agent(agent, config):
    response = agent.invoke(
        {
            "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
            "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John."
        },
        config=config,
    )
    return response

In [129]:
config = make_thread_config()

agent = initialize_new_agent(temperature=0.0)
response = invoke_agent(agent, config)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (call_024cf165-6ce7-4e0c-835c-4892cbb68e65)
 Call ID: call_024cf165-6ce7-4e0c-835c-4892cbb68e65
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (call_2305dc54-b26f-4ff0-a9e7-b0f7e5a466f5)
 Call ID: call_2305dc54-b26f-4ff0-a9e7-b0f7e5a466f5
  Args:
    body: Hi John,

No problem at all. Let's reschedule. Please let me know what time works best for you tomorrow or if you'd prefer to meet another day.

Best regards,
Seán


In [130]:
response["__interrupt__"]

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem at all. Let's reschedule. Please let me know what time works best for you tomorrow or if you'd prefer to meet another day.\n\nBest regards,\nSeán"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem at all. Let\'s reschedule. Please let me know what time works best for you tomorrow or if you\'d prefer to meet another day.\\n\\nBest regards,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='aa85ff031b051f7b4c67b20156ead754')]

In [131]:
pp(response["__interrupt__"][0].value)

{
    'action_requests': [
        {
            'args': {
                'body': "Hi John,\n\nNo problem at all. Let's reschedule. Please let me know what time works best for you tomorrow or if you'd prefer to meet another day.\n\nBest regards,\nSeán",
            },
            'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem at all. Let\'s reschedule. Please let me know what time works best for you tomorrow or if you\'d prefer to meet another day.\\n\\nBest regards,\\nSeán"}',
            'name': 'send_email',
        },
    ],
    'review_configs': [
        {
            'action_name': 'send_email',
            'allowed_decisions': ['approve', 'edit', 'reject', 'respond'],
        },
    ],
}


## Approve

In [132]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config, # Same thread ID to resume the paused conversation
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (call_024cf165-6ce7-4e0c-835c-4892cbb68e65)
 Call ID: call_024cf165-6ce7-4e0c-835c-4892cbb68e65
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (call_2305dc54-b26f-4ff0-a9e7-b0f7e5a466f5)
 Call ID: call_2305dc54-b26f-4ff0-a9e7-b0f7e5a466f5
  Args:
    body: Hi John,

No problem at all. Let's reschedule. Please let me know what time works best for you tomorrow or if you'd prefer to meet another day.

Best regards,
Seán
================================= Tool Mes

## Reject

In [133]:
config = make_thread_config()

agent = initialize_new_agent(temperature=0.5)
response = invoke_agent(agent, config)

In [134]:

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No!! - Request rejected!."
                }
            ],
        }
    ), 
    config=config,
    )   

In [135]:

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (call_8d90c43f-55ae-4d30-bb2c-fabdfb79dacd)
 Call ID: call_8d90c43f-55ae-4d30-bb2c-fabdfb79dacd
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (call_e31bda22-d236-4c8c-b200-cf84427825b6)
 Call ID: call_e31bda22-d236-4c8c-b200-cf84427825b6
  Args:
    body: Hi John,

No problem at all. Let's reschedule. Please let me know what time works best for you tomorrow or if you'd prefer to meet another day.

Best regards,
Seán
================================= Tool Mes

## Edit

In [136]:
config = make_thread_config()

agent = initialize_new_agent(temperature=1.0)
response = invoke_agent(agent, config)

In [137]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        "name": "send_email",  # Tool name to call. # Will usually be the same as the original action.
                        
                        # Arguments to pass to the tool. Updated body will generally come from a UI where 
                        # user has edited the email body
                        "args": {"body": "Let me see what we need to do to reschedule the meeting."},
                    }
                }
            ]
        }
    ), 
    config=config, # Same thread ID to resume the paused conversation
    )   

In [138]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (call_53503358-b62b-407c-94c6-1d40ced4dfce)
 Call ID: call_53503358-b62b-407c-94c6-1d40ced4dfce
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (call_f5fb3dfa-69f6-42af-9007-4117ab361ae4)
 Call ID: call_f5fb3dfa-69f6-42af-9007-4117ab361ae4
  Args:
    body: Let me see what we need to do to reschedule the meeting.
================================= Tool Message =================================
Name: send_email

Email sent with body: Let me see what we need to d